# Capstone — Which Pages Deserve a Look First?

**A ranked review queue for a content team with thousands of pages.**
Lane: Refresh / Content Opportunity Scoring · decision-support only.

This notebook mirrors the deployed research paper section-for-section:
**https://sukuna310.github.io/MY-Flyrank-ml-internship-starter/**

**Abstract.** A content team managing thousands of pages across dozens of clients has one editor who can review only a handful of pages per cycle — and no defensible order for which ones to look at first. We build a five-feature logistic regression on 151,981 page-level rows from the FlyRank internship warehouse (month=2026-03), labeling 77,400 pages (≥100 first-half impressions) by whether their second-half March impressions fell below their first-half — a within-month proxy, not a forecast. Validation is GroupKFold(5) by client, so no client appears in both train and test. The model clears random ranking modestly (grouped AUC 0.5902 vs 0.5000; grouped P@50 0.4800 vs 0.4080 random) and does **not** separate from the hand-written rule baseline (P@50 0.4360, same folds); the only reason-code group that clears the base rate after the priority contest is low CTR (decline 0.5072 below median vs 0.3804 above). The output is a reading order with per-page reason codes, not an automation: no refresh action is assigned anywhere, because the best lift among the queue's reason-code patterns over the 0.4438 base rate is +0.0634.

Every number below is **re-measured in this notebook** from the local warehouse cache — the receipts in the final cell check them against the paper. No client names, client domains/URLs, or private queries appear anywhere: pseudonymized hashes and aggregates only.

## Setup

The warehouse slice is cached locally (one full scan, cached — never re-downloaded at run time). A fresh kernel needs the packages below; a READ token is never pasted in a cell.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: /Users/abdelrahmankhaled/Documents/ML FlyRank ML Internship Folder/MY-Flyrank-ml-internship-starter/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- Setup: sources, windows, engine (the same data contract as w03 / w06 / w07) ---
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score

def find_root():
    p = Path.cwd()
    for _ in range(4):                      # work/notebooks -> repo root, wherever jupyter runs
        if (p / "AGENTS.md").exists():
            return p
        p = p.parent
    return Path(".")

ROOT = find_root()
CACHE = ROOT / "work" / "outputs" / "warehouse_cache"
for needed in ["fact_2026-03.parquet", "dim_content.parquet", "dim_clients.parquet"]:
    if not (CACHE / needed).exists():
        raise SystemExit(f"warehouse cache missing {needed} - see skills/flyrank/flyrank-data "
                         "(request access to FlyRank/internship-warehouse; a READ token is never pasted in a cell).")

FACT        = f"read_parquet('{CACHE}/fact_2026-03.parquet')"
DIM_CONTENT = f"read_parquet('{CACHE}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{CACHE}/dim_clients.parquet')"

DECISION_DATE = "2026-03-15"
FEAT_LO, FEAT_HI = "2026-03-01", "2026-03-15"   # feature window: knowable at the decision moment
OUT_LO,  OUT_HI  = "2026-03-16", "2026-03-31"   # outcome window: label only, never a feature

con = duckdb.connect()
print("source cache  :", CACHE.name)
print("decision date :", DECISION_DATE)
print("feature window:", FEAT_LO, "->", FEAT_HI)
print("outcome window:", OUT_LO, "->", OUT_HI)
print("versions      : pandas", pd.__version__, "| numpy", np.__version__,
      "| sklearn", __import__("sklearn").__version__, "| duckdb", duckdb.__version__)

source cache  : warehouse_cache
decision date : 2026-03-15
feature window: 2026-03-01 -> 2026-03-15
outcome window: 2026-03-16 -> 2026-03-31
versions      : pandas 3.0.5 | numpy 2.5.2 | sklearn 1.9.0 | duckdb 1.5.5


## 1. Question

**The decision:** an editor opens a queue on the morning of 2026-03-15 and asks *"which pages do I look at first, and why?"* One editor, a few reviews per cycle, 77,400 labeled pages — the binding constraint is the editor's calendar, not the data.

**Why not a plain sort?** Sorting by impressions descending puts the largest pages on top whether or not anything is wrong with them, and buries small failing pages. An editor's memory is a real signal but cannot be handed to a colleague or checked six weeks later.

**Why ML at all?** Only as a *reading order*: five feature-window signals, tangled, scored against an outcome that already occurred inside the same month. The claim is decision-support, never prediction — the score does not forecast April. A wrong call costs a wasted editor-hour, which is why nothing destructive is ever automated from it (see the no-go list in §6).

## 2. Data

| Item | Value |
|---|---|
| Release | FlyRank internship warehouse, `month=2026-03` (mid-panel month; the `_sample` table is the panel's last month and is never touched) |
| Tables | `fact_content_daily_performance` (daily GSC facts), `dim_content` (creation date), `dim_clients` (coverage check only) |
| Daily rows in the month | 9,841,378 → 3,611,061 survive `gsc_data_available IS TRUE` |
| Feature-frame rows | 151,981 (one row per client × page, 15-day feature window) |
| Labeled rows | 77,400 (≥100 first-half impressions), 38 clients, base rate 0.4438 |

**Excluded, deliberately:** GA4 columns (three-valued availability flag, zero-filled early history — a second, messier measurement system); the `_sample` table (it would peek at the test window); product flags (must not be rebuilt as features); `trend_direction` / `trend_pct` (label-derived in the starter data — never features).

In [3]:
# --- Slice size + availability: the filter that defines the population ---
counts = con.sql(f"""
  SELECT COUNT(*)                                            AS month_rows,
         COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_survive,
         MIN(report_date)                                    AS first_date,
         MAX(report_date)                                    AS last_date,
         COUNT(DISTINCT client_hash_id)                      AS n_clients
  FROM {FACT}
""").df()
print(counts.to_string(index=False))
print("\nOnly gsc_data_available IS TRUE rows are read as real measurements.")
print("GA4 columns are excluded by design (see the markdown above).")

 month_rows  gsc_survive first_date  last_date  n_clients
    9841378      3611061 2026-03-01 2026-03-31         55

Only gsc_data_available IS TRUE rows are read as real measurements.
GA4 columns are excluded by design (see the markdown above).


In [4]:
# --- The five-feature frame: feature window only; the label comes from the outcome window ---
# Same SQL as w03 / w06 / w07, with the stable ORDER BY that makes every number reproducible.
FEATURE_CTE = f"""
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS f_impressions,
    SUM(gsc_clicks) AS f_clicks,
    AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS f_avg_position,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS f_active_days
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{FEAT_LO}' AND DATE '{FEAT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
"""
OUTCOME_CTE = f"""
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS t_impressions
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
"""
frame_sql = f"""
WITH feat AS ({FEATURE_CTE}), outcome AS ({OUTCOME_CTE})
SELECT
  feat.client_hash_id, feat.content_hash_id,
  LN(1 + feat.f_impressions)                             AS f_log_impressions,
  feat.f_clicks::DOUBLE / NULLIF(feat.f_impressions, 0) AS f_ctr,
  feat.f_avg_position, feat.f_active_days,
  LN(1 + DATEDIFF('day', dim.content_created_date, DATE '{DECISION_DATE}')) AS f_log_age_days,
  CASE WHEN feat.f_impressions >= 100
       THEN (outcome.t_impressions < feat.f_impressions)::INT END AS is_visibility_loss
FROM feat
LEFT JOIN outcome USING (client_hash_id, content_hash_id)
LEFT JOIN {DIM_CONTENT} dim USING (content_hash_id)
ORDER BY feat.client_hash_id, feat.content_hash_id   -- stable row order, so every number is reproducible
"""
frame = con.sql(frame_sql).df()
FEATURES = ["f_log_impressions", "f_ctr", "f_avg_position", "f_active_days", "f_log_age_days"]

print(f"frame rows {len(frame):,} | labeled rows {int(frame['is_visibility_loss'].notna().sum()):,}")
print(f"label rate among labeled rows: {frame['is_visibility_loss'].mean():.4f}")
print("\nmissingness per feature (fill policy: NaN -> 0, applied only inside the model matrix):")
for c in FEATURES:
    print(f"  {c:20s} missing = {int(frame[c].isna().sum()):>6}")
frame.head(3)

frame rows 151,981 | labeled rows 77,400
label rate among labeled rows: 0.4438

missingness per feature (fill policy: NaN -> 0, applied only inside the model matrix):
  f_log_impressions    missing =      0
  f_ctr                missing =      0
  f_avg_position       missing =      0
  f_active_days        missing =      0
  f_log_age_days       missing =      0


,client_hash_id,content_hash_id,f_log_impressions,f_ctr,f_avg_position,f_active_days,f_log_age_days,is_visibility_loss
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,4.787492,0.008403,12.639599,15,5.075174,0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,3.526361,0.000000,9.225529,6,5.075174,<NA>
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,4.290459,0.000000,8.094074,15,5.075174,<NA>


## 3. Methodology

- **Label (observed, not rule-defined):** `is_visibility_loss = 1` when second-half March impressions < first-half March impressions, computed only for pages with ≥100 first-half impressions. A within-month proxy — **not** a future-window forecast.
- **Features (5, all knowable on 2026-03-15):** `f_log_impressions`, `f_ctr`, `f_avg_position`, `f_active_days`, `f_log_age_days`.
- **Model:** `LogisticRegression(max_iter=500, C=1.0)` — no scaler, the exact w03/w05/w06 estimator. A linear model's coefficients are readable reason codes; an ensemble would score the same task without them.
- **Validation:** `GroupKFold(5)` by `client_hash_id` — no client in both train and test. A seeded random row split is quoted **for contrast only**: the gap between the two is client memorisation, not generalisation.
- **Baseline:** the Week-4 hand rule (fits nothing), scored on the *same* grouped folds.
- **Leakage checks:** timeline assertions + a deliberate leak test — add the outcome-window sum as a feature, watch AUC hit ~1.0, remove it, confirm the honest number survives.

In [5]:
# --- Model matrix, base rate, timeline checks, and the deliberate leakage test ---
labeled = frame.dropna(subset=["is_visibility_loss"]).copy()
y      = labeled["is_visibility_loss"].astype(int).values
groups = labeled["client_hash_id"].values
X      = labeled[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
BASE_RATE = float(y.mean())
MODEL = LogisticRegression(max_iter=500, C=1.0)      # exactly the w03 / w05 / w06 estimator

# LBFGS can raise transient overflow RuntimeWarnings on this frame (f_ctr ~ 0 on most pages).
# w06 cell 4 measured n_iter_ 6-8 on the same estimator/frame, well inside max_iter=500;
# non-convergence would instead appear as ConvergenceWarning. Silenced on that evidence.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn.*")

print(f"labeled rows {len(labeled):,} | clients {labeled['client_hash_id'].nunique()} | base rate {BASE_RATE:.4f}")

# Timeline: the feature CTE cannot see the outcome window.
assert FEAT_HI <= DECISION_DATE and OUT_LO > FEAT_HI
assert not (OUT_LO in FEATURE_CTE or OUT_HI in FEATURE_CTE)
print("timeline check: features end on/before the decision date; the label window starts after it. OK")

# Deliberate leak test (w03): the outcome-window sum is the label's own input.
RAND3 = StratifiedKFold(3, shuffle=True, random_state=0)   # seeded shuffle: reproducible AND genuinely random
def quick_auc(cols, df=None):
    df = labeled if df is None else df
    Xm = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    return round(cross_val_score(MODEL, Xm, y, cv=RAND3, scoring="roc_auc").mean(), 4)

leak = con.sql(f"""
  SELECT client_hash_id, content_hash_id, LN(1 + SUM(gsc_impressions)) AS leak_future_imps
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
""").df()
trap = labeled.merge(leak, on=["client_hash_id", "content_hash_id"], how="left")
honest = quick_auc(FEATURES)
leaky  = quick_auc(FEATURES + ["leak_future_imps"], df=trap)
trap.drop(columns=["leak_future_imps"], inplace=True)
after  = quick_auc(FEATURES)
print(f"\nquick AUC, 5 honest features     : {honest}")
print(f"quick AUC, + outcome-window leak : {leaky}   <- near-perfect: the trap, means nothing")
print(f"quick AUC, leak column removed   : {after}   <- the honest number survives")
print("leak column present in any final feature list?", "leak_future_imps" in FEATURES)

labeled rows 77,400 | clients 38 | base rate 0.4438
timeline check: features end on/before the decision date; the label window starts after it. OK



quick AUC, 5 honest features     : 0.6262
quick AUC, + outcome-window leak : 1.0   <- near-perfect: the trap, means nothing
quick AUC, leak column removed   : 0.6262   <- the honest number survives
leak column present in any final feature list? False


## 4. Results (vs baseline)

The honest table — same estimator, same five features, same rows; **only the split changes**. The grouped number is the only one that carries a claim.

| Method | Split | AUC | P@50 | Base rate |
|---|---|---|---|---|
| Random ordering (reference) | grouped, same folds | 0.5000 | 0.4080 | 0.4438 |
| Week-4 hand rule (baseline) | grouped, same folds | — | 0.4360 | 0.4438 |
| Logistic regression (this work) | grouped, same folds | 0.5902 | 0.4800 | 0.4438 |
| Logistic regression (contrast only) | random rows, seeded | 0.6262 | 0.5600 | 0.4438 |

The table quotes the paper's recorded comparator values; the recorded +0.0440 P@50 gap sits **inside the rule's recorded 0.20–0.58 fold spread** → *unseparated, not a win*. The code below re-measures every cell and prints the fresh values.

**Version sensitivity (measured, not hidden):** the model's own metrics, base rate, counts and all pattern rates reproduce exactly, but the two *comparator* numbers — random-ordering P@50 (recorded 0.4080, fresh 0.4320) and hand-rule P@50 (recorded 0.4360, fresh 0.4520) — move by ~0.02 on a fresh run. Random ordering depends on the seeded permutation; the rule has only three score levels, so `np.argsort` tie-breaking decides which tied rows enter the top 50. The rule's fold-top (0.58 recorded, 0.60 fresh) and the model-minus-rule gap (+0.0440 recorded, +0.0280 fresh) move with the same tie-breaks. Both comparators land **below the model's 0.4800** under either measurement, and the gap stays inside the rule's fold spread. The conclusion (modest signal, no separation from the rule) is unchanged; the third decimal of a comparator is not a result.

In [6]:
# --- AUC: same estimator, only the split changes ---
auc_rand  = float(cross_val_score(MODEL, X, y, cv=RAND3, scoring="roc_auc").mean())
gkf       = GroupKFold(5)
grp_folds = cross_val_score(MODEL, X, y, cv=gkf, groups=groups, scoring="roc_auc")
auc_grp   = float(grp_folds.mean())
print(f"random row split (seeded) AUC : {auc_rand:.4f}   <- contrast only")
print(f"grouped by client AUC         : {auc_grp:.4f}   folds {np.round(grp_folds, 4)}")
print(f"gap given back by grouping    : {auc_grp - auc_rand:+.4f}  (client memorisation, not generalisation)")

# --- P@50 ranked WITHIN each fold: scores from different fold-models are not on one scale ---
def per_fold_p_at_k(splitter, grp=None, k=50, seed=0):
    model_hits, random_hits = [], []
    splits = splitter.split(X, y, grp) if grp is not None else splitter.split(X, y)
    for tr, te in splits:
        if len(te) < k:
            continue
        m = MODEL.fit(X[tr], y[tr])
        model_hits.append(y[te][np.argsort(-m.predict_proba(X[te])[:, 1])[:k]].mean())
        random_hits.append(y[te][np.random.RandomState(seed).permutation(len(te))[:k]].mean())
    return float(np.mean(model_hits)), float(np.mean(random_hits))

p_rand_m, p_rand_r = per_fold_p_at_k(RAND3)
p_grp_m,  p_grp_r  = per_fold_p_at_k(gkf, groups)

# --- Week-4 hand rule baseline: fits nothing, scored on the same grouped folds ---
rule = labeled.copy(); rule["rule_score"] = 0.0
c_a = (rule["f_avg_position"] <= 5) & (rule["f_ctr"] < rule.loc[rule["f_avg_position"] <= 5, "f_ctr"].median())
c_b = (rule["f_log_impressions"] >= rule["f_log_impressions"].median()) & \
      (rule["f_log_age_days"] >= rule["f_log_age_days"].median())
c_c = (rule["f_log_impressions"] >= rule["f_log_impressions"].median()) & \
      (rule["f_ctr"] < rule["f_ctr"].median())
rule.loc[c_a, "rule_score"] += 2
rule.loc[c_b & ~c_a, "rule_score"] += 1
rule.loc[c_c & ~c_a & ~c_b, "rule_score"] += 1
rule_rs = rule["rule_score"].to_numpy()
rule_folds = [y[te][np.argsort(-rule_rs[te])[:50]].mean()
              for _, te in gkf.split(X, y, groups) if len(te) >= 50]
p_rule = float(np.mean(rule_folds))

results = pd.DataFrame([
    ["random ordering (reference)", "grouped, same folds", 0.5000, p_grp_r, BASE_RATE],
    ["Week-4 hand rule (baseline)", "grouped, same folds", np.nan, p_rule, BASE_RATE],
    ["logistic regression (this work)", "grouped, same folds", auc_grp, p_grp_m, BASE_RATE],
    ["logistic regression (contrast)", "random rows, seeded", auc_rand, p_rand_m, BASE_RATE],
], columns=["method", "split", "AUC", "P@50", "base rate"])
print("\n=== the honest table, re-measured ===")
print(results.round(4).to_string(index=False))
print(f"\nrule fold spread {min(rule_folds):.2f}-{max(rule_folds):.2f}: the model's +{p_grp_m - p_rule:.4f} gap "
      "sits inside it -> unseparated, not a win.")
print(f"one grouped fold lands below chance ({grp_folds.min():.4f}); the fold SD is {grp_folds.std():.4f}. Quoted, not hidden.")

random row split (seeded) AUC : 0.6262   <- contrast only
grouped by client AUC         : 0.5902   folds [0.5988 0.6214 0.4666 0.6547 0.6096]
gap given back by grouping    : -0.0360  (client memorisation, not generalisation)



=== the honest table, re-measured ===
                         method               split    AUC  P@50  base rate
    random ordering (reference) grouped, same folds 0.5000 0.432     0.4438
    Week-4 hand rule (baseline) grouped, same folds    NaN 0.452     0.4438
logistic regression (this work) grouped, same folds 0.5902 0.480     0.4438
 logistic regression (contrast) random rows, seeded 0.6262 0.560     0.4438

rule fold spread 0.20-0.60: the model's +0.0280 gap sits inside it -> unseparated, not a win.
one grouped fold lands below chance (0.4666); the fold SD is 0.0646. Quoted, not hidden.


## 5. Limitations — what this work cannot claim

- **One month, one decision date, one label definition.** A March-2026 model, not a general one.
- **Within-month proxy, not future decline.** The label never sees anything after 2026-03-31; the score orders an already-observed population, it does not forecast.
- **High-traffic pages only.** 77,400 of 151,981 rows carry a label; the unlabeled half is systematically thinner (median active days 6 vs 15). Nothing here reads onto low-traffic pages.
- **The labeled set conditions on being observed after the decision date.** 140 pages met the 100-impression threshold but had no outcome-window row; with a NULL outcome they are unlabeled (77,540 eligible → 77,400 labeled). Fewer than 0.2% of eligible pages, inherited from w03/w06/w07 — disclosed, not silently dropped.
- **38 uneven clients** (median 485 rows, max 17,610; largest ≈22.8% of the queue). Pooled numbers are not per-client numbers.
- **Modest signal.** Grouped AUC 0.5902; one fold below chance. The model does not beat the rule.
- **No time-aware split exists** — the largest untested assumption. A monitoring plan (w07 §4) is offered in lieu of a validation result.

Words used: *observed, measured, directional, decision-support, associated with*. Causal and forecast words appear only inside negations and no-go statements — never as claims about a page.

In [7]:
# --- The limits as measurements, not prose (same checks as w07 §2) ---
unlabeled  = frame[frame["is_visibility_loss"].isna()]
per_client = labeled.groupby("client_hash_id").size()
limits = pd.DataFrame([
    ["population", f"{len(labeled):,} of {len(frame):,} frame rows labeled ({len(labeled)/len(frame):.1%})",
     "the queue cannot speak about the other half"],
    ["selection effect", f"labeled median active_days {labeled['f_active_days'].median():.0f} vs "
                         f"unlabeled {unlabeled['f_active_days'].median():.0f}",
     "labeled pages are already the active ones"],
    ["base rate", f"{BASE_RATE:.4f}", "any precision number needs this next to it"],
    ["client concentration", f"{labeled['client_hash_id'].nunique()} clients, median {per_client.median():,.0f} rows, "
                             f"max {per_client.max():,} ({per_client.max()/len(labeled):.1%} of the queue)",
     "pooled numbers are not per-client numbers"],
    ["windows", f"features {FEAT_LO}..{FEAT_HI}, label {OUT_LO}..{OUT_HI}, one decision date",
     "a March-2026 model, not a general one"],
], columns=["limit", "measured here", "consequence for the reader"])
print(limits.to_string(index=False))

               limit                                                                    measured here                  consequence for the reader
          population                                     77,400 of 151,981 frame rows labeled (50.9%) the queue cannot speak about the other half
    selection effect                                     labeled median active_days 15 vs unlabeled 6   labeled pages are already the active ones
           base rate                                                                           0.4438  any precision number needs this next to it
client concentration                     38 clients, median 485 rows, max 17,610 (22.8% of the queue)   pooled numbers are not per-client numbers
             windows features 2026-03-01..2026-03-15, label 2026-03-16..2026-03-31, one decision date       a March-2026 model, not a general one


## 6. Ranked recommendations — the action playbook

**Priority order for the editor** (each row's measured basis is re-derived below):

1. **Look at low-CTR pages first** — check intent match, not a verdict. Below-median CTR declines 0.5072 vs 0.3804 above: the largest measured gap, and the only reason code whose assigned group clears the base rate. *Directional.*
2. **Check position trend by eye** — `f_avg_position` is a 15-day average, not a slope; its permutation importance is indistinguishable from zero (w06). *Low confidence.*
3. **Monitor high-volume pages** — standalone lift only +0.0191. *Weak.*
4. **Do NOT refresh `low_activity_concentration` pages** — they decline *least* (0.1251 vs 0.4948 for 15/15-day pages). The negative result is the finding.
5. **Do not auto-decide anything from this score** — the best reason-code lift +0.0634 cannot carry a destructive action.

**No-go list:** no auto-delete/noindex/redirect; no auto-refresh/rewrite/republish; no action below the 100-impression threshold; no reading the score as a prediction; no refresh of healthy-looking pages (they decline *below* base rate — measured below); no client-facing "risk"/"health" grade.

The age heuristic ran **backwards**: the oldest quartile declines least (0.3707), so `stale_high_traffic` was dropped and assigned to zero pages.

In [8]:
# --- Reason codes + the ranked queue (the exact w07 logic, re-measured here) ---
queue_model = MODEL.fit(X, y)                       # full-frame fit: the queue's score is IN-SAMPLE;
labeled["predicted_probability"] = queue_model.predict_proba(X)[:, 1]   # ranking quality is only ever quoted out-of-fold (§4)

q = labeled.copy()
q["f_active_days"] = q["f_active_days"].fillna(0)
POS, CTR_MED = q["f_avg_position"].median(), q["f_ctr"].median()
VOL_T, ACT_MAX = q["f_log_impressions"].quantile(2 / 3), 10
CODE_ORDER = ["low_ctr_more_decline", "high_volume_tilt", "poor_position_drift",
              "low_activity_concentration", "no_measured_pattern"]
ACTION_OF = {"low_ctr_more_decline": "ctr_review", "high_volume_tilt": "monitor",
             "poor_position_drift": "position_check",
             "low_activity_concentration": "monitor_low_priority",
             "no_measured_pattern": "monitor_low_priority"}

# Each cut is justified by its STANDALONE measured rate (every row the pattern matches),
# not by its coefficient and not by its post-contest group.
PATTERNS = {"low_ctr_more_decline": q["f_ctr"] < CTR_MED,
            "high_volume_tilt": q["f_log_impressions"] >= VOL_T,
            "poor_position_drift": q["f_avg_position"] > POS,
            "low_activity_concentration": q["f_active_days"] <= ACT_MAX}
standalone = pd.DataFrame([
    {"reason_code": c, "n": int(m.sum()),
     "decline_rate": float(q.loc[m, "is_visibility_loss"].mean()),
     "lift_vs_base_rate": float(q.loc[m, "is_visibility_loss"].mean()) - BASE_RATE,
     "action": ACTION_OF[c]} for c, m in PATTERNS.items()])
standalone = standalone.sort_values("lift_vs_base_rate", ascending=False).reset_index(drop=True)
print("=== each reason code measured STANDALONE (priority order = this order) ===")
print(standalone.round(4).to_string(index=False))

# One primary code per row, highest measured lift wins: writes go lowest-lift-first so the best survives.
q["reason_code"] = "no_measured_pattern"
q["action"] = ACTION_OF["no_measured_pattern"]
for c in reversed(list(standalone["reason_code"])):
    q.loc[PATTERNS[c], ["reason_code", "action"]] = [c, ACTION_OF[c]]
q = q.sort_values(["predicted_probability", "client_hash_id", "content_hash_id"],
                  ascending=[False, True, True]).reset_index(drop=True)
q.insert(0, "rank", np.arange(1, len(q) + 1))

by_code = (q.groupby("reason_code")
             .agg(n=("is_visibility_loss", "size"), decline_rate=("is_visibility_loss", "mean"))
             .reindex(CODE_ORDER))
by_code["n"] = by_code["n"].fillna(0).astype(int)
by_code["lift_vs_base_rate"] = by_code["decline_rate"] - BASE_RATE
by_code["share_of_queue"] = by_code["n"] / len(q)
by_code["action"] = pd.Series(ACTION_OF)
print("\n=== after the priority contest: what the queue actually produced ===")
print(by_code.round(4).to_string())
above = by_code.index[by_code["lift_vs_base_rate"] > 0].tolist()
print(f"\ncodes whose ASSIGNED group clears the base rate {BASE_RATE:.4f}: {above}")
print("no code is assigned review_for_refresh: the best reason-code lift "
      f"{standalone['lift_vs_base_rate'].max():+.4f} is not enough to tell an editor to change a page.")

# The headline split, both sides (the largest measured gap in the frame).
ctr_lo_rate = float(q.loc[q["f_ctr"] < CTR_MED, "is_visibility_loss"].mean())
ctr_hi_rate = float(q.loc[q["f_ctr"] >= CTR_MED, "is_visibility_loss"].mean())
print(f"\nCTR median {CTR_MED:.5f}: below -> decline {ctr_lo_rate:.4f} | above -> {ctr_hi_rate:.4f} "
      f"(gap {ctr_lo_rate - ctr_hi_rate:+.4f})")

# The activity reversal, read directly (more active days -> MORE decline).
act_low_rate = float(q.loc[q["f_active_days"] <= ACT_MAX, "is_visibility_loss"].mean())
act_15_rate  = float(q.loc[q["f_active_days"] == 15, "is_visibility_loss"].mean())
n15 = int((q["f_active_days"] == 15).sum())
print(f"active <= {ACT_MAX} days -> decline {act_low_rate:.4f} | 15/15 days -> {act_15_rate:.4f} "
      f"| {n15/len(q):.1%} of pages sit at 15/15 (why the cut is absolute, not rank-based)")

# The age heuristic, measured: oldest declines LEAST -> stale_high_traffic assigned to 0 pages.
ageable = q["f_log_age_days"].notna()
q["age_quartile"] = pd.NA
q.loc[ageable, "age_quartile"] = pd.qcut(q.loc[ageable, "f_log_age_days"], 4,
                                          labels=["Q1_newest", "Q2", "Q3", "Q4_oldest"])
age = (q[ageable].groupby("age_quartile", observed=True)["is_visibility_loss"]
         .agg(["size", "mean"]).reset_index())
age_oldest_rate = float(age["mean"].iloc[-1])
print("\nage quartiles (newest -> oldest):", [round(v, 4) for v in age["mean"]])
print(f"oldest quartile declines {age_oldest_rate:.4f} vs base rate {BASE_RATE:.4f} -> 'older declines more' is FALSE here.")

# False-positive risk: healthy-looking pages in the queue decline BELOW base rate.
q["looks_healthy"] = (q["f_avg_position"] <= 5) & (q["f_ctr"] >= CTR_MED)
n_healthy = int(q["looks_healthy"].sum())
healthy_rate = float(q.loc[q["looks_healthy"], "is_visibility_loss"].mean())
print(f"\nhealthy-looking pages (position <= 5 AND ctr >= median): {n_healthy:,} of {len(q):,}, "
      f"decline {healthy_rate:.4f} (lift {healthy_rate - BASE_RATE:+.4f}) -> never auto-refresh them.")

print("\n=== top 10 of the ranked queue (pseudonymized hashes, truncated) ===")
print(q.head(10)[["rank", "content_hash_id", "predicted_probability", "reason_code", "action",
                  "f_active_days", "f_ctr", "f_avg_position", "is_visibility_loss"]]
        .assign(content_hash_id=lambda d: d["content_hash_id"].str[:10] + "...").round(5).to_string(index=False))

=== each reason code measured STANDALONE (priority order = this order) ===
               reason_code     n  decline_rate  lift_vs_base_rate               action
      low_ctr_more_decline 38695        0.5072             0.0634           ctr_review
          high_volume_tilt 25820        0.4629             0.0191              monitor
       poor_position_drift 38700        0.4620             0.0182       position_check
low_activity_concentration  4638        0.1251            -0.3187 monitor_low_priority

=== after the priority contest: what the queue actually produced ===
                                n  decline_rate  lift_vs_base_rate  share_of_queue                action
reason_code                                                                                             
low_ctr_more_decline        38695        0.5072             0.0634          0.4999            ctr_review
high_volume_tilt            16562        0.3855            -0.0583          0.2140               monitor


## 7. Artifacts the paper embeds

Reproducibility is **measured, not asserted**: the frame SQL is re-run and compared feature-by-feature. The frame does not reproduce bit-for-bit — `f_avg_position` is a parallel DuckDB `AVG`, and parallel float addition is not associative (~4e-14 noise). Everything reported is many orders of magnitude above that floor. The two figures below are the paper's Figure 1 and Figure 2; the queue CSV and metrics JSON are gitignored artifacts (code and reports go in git, data does not).

In [9]:
# --- Reproducibility check + the paper's two figures + exports ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR, FIG_DIR = ROOT / "work" / "outputs", ROOT / "work" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

frame2 = con.sql(frame_sql).df()
same_rows = (frame2[["client_hash_id", "content_hash_id"]].values
             == frame[["client_hash_id", "content_hash_id"]].values).all()
feat_repro = {c: float(np.nanmax(np.abs(frame2[c].to_numpy(dtype=float)
                                          - frame[c].to_numpy(dtype=float)))) for c in FEATURES}
worst = max(feat_repro.values())
print("=== reproducibility of the frame (re-ran the same SQL, same stable ORDER BY) ===")
print(f"same rows in the same order : {bool(same_rows)}")
print("max abs diff per feature    : " + ", ".join(f"{c} {v:.1e}" for c, v in feat_repro.items()))
print(f"worst difference {worst:.1e} -> within 1e-12? {worst <= 1e-12}  (parallel-AVG float noise, not a data change)")

# Figure 1 — decline rate by score decile (in-sample consistency check, NOT a performance claim).
q["score_decile"] = pd.qcut(q["predicted_probability"], 10, labels=False, duplicates="drop") + 1
dec = (q.groupby("score_decile")
         .agg(n=("is_visibility_loss", "size"), decline_rate=("is_visibility_loss", "mean")).reset_index())
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.bar(dec["score_decile"], dec["decline_rate"], color="#4C6EF5", width=0.72, label="measured decline rate")
ax.axhline(BASE_RATE, color="#868E96", ls="--", lw=1.4, label=f"base rate {BASE_RATE:.4f}")
ax.set_xlabel("score decile (10 = highest predicted probability)")
ax.set_ylabel("is_visibility_loss rate")
ax.set_title("Queue head is where decline concentrates - in-sample, March 2026 slice", fontsize=11)
ax.text(0.02, 0.94, f"decile 1 {dec['decline_rate'].iloc[0]:.4f}  ->  decile 10 "
                    f"{dec['decline_rate'].iloc[-1]:.4f}   (n={len(q):,} labeled rows, 38 clients)",
        transform=ax.transAxes, fontsize=8.5, va="top", color="#495057")
ax.set_xticks(dec["score_decile"])
ax.legend(frameon=False, fontsize=9, loc="lower right")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "queue_decline_profile.png", dpi=150)
plt.close(fig)
print(f"\nwrote work/figures/queue_decline_profile.png  (decile 1 {dec['decline_rate'].iloc[0]:.4f} -> "
      f"decile 10 {dec['decline_rate'].iloc[-1]:.4f}; the out-of-fold numbers to quote are in §4)")

# Figure 2 — reason codes: share of queue (left) and measured decline rate vs base rate (right).
fig2, (axL, axR) = plt.subplots(1, 2, figsize=(11.0, 4.2))
bc = by_code.reset_index()
cols = ["#4C6EF5" if v > BASE_RATE else "#ADB5BD" for v in bc["decline_rate"]]
axL.bar(bc["reason_code"], bc["share_of_queue"], color=cols, width=0.68)
for x, share in enumerate(bc["share_of_queue"]):
    axL.text(x, share + 0.006, f"{share:.1%}", ha="center", fontsize=8.5, color="#495057")
axL.set_ylabel("share of the 77,400-row queue")
axL.set_title("How much of the queue each reason code holds", fontsize=10.5)
axL.set_xticks(np.arange(len(bc)))
axL.set_xticklabels(bc["reason_code"], rotation=32, ha="right", fontsize=8.5)
axL.set_ylim(0, max(bc["share_of_queue"]) * 1.18)
axR.bar(bc["reason_code"], bc["decline_rate"], color=cols, width=0.68)
axR.axhline(BASE_RATE, color="#868E96", ls="--", lw=1.4, label=f"base rate {BASE_RATE:.4f}")
for x, v in enumerate(bc["decline_rate"]):
    axR.text(x, v + 0.008, f"{v:.3f}", ha="center", fontsize=8.5, color="#495057")
axR.set_ylabel("measured is_visibility_loss rate")
axR.set_ylim(0, max(bc["decline_rate"]) * 1.20)
axR.set_title("Measured decline rate of the pages each code filed", fontsize=10.5)
axR.set_xticks(np.arange(len(bc)))
axR.set_xticklabels(bc["reason_code"], rotation=32, ha="right", fontsize=8.5)
axR.legend(frameon=False, fontsize=8.5, loc="upper right")
for ax in (axL, axR):
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", alpha=0.22)
fig2.tight_layout()
fig2.savefig(FIG_DIR / "reason_code_breakdown.png", dpi=150)
plt.close(fig2)
print("wrote work/figures/reason_code_breakdown.png  (blue = above base rate, grey = at or below;")
print(" the grey bars are the honest half of the figure, not a defect)")

# Exports (gitignored): the ranked queue + this notebook's re-measured metrics.
QUEUE_COLS = ["rank", "client_hash_id", "content_hash_id", "predicted_probability",
              "reason_code", "action"] + FEATURES + ["is_visibility_loss", "score_decile"]
q_out = q.copy()
q_out["predicted_probability"] = q_out["predicted_probability"].round(6)
for c in FEATURES:
    q_out[c] = q_out[c].round(6)
q_out[QUEUE_COLS].to_csv(OUT_DIR / "action_playbook_queue.csv", index=False)
print(f"\nwrote work/outputs/action_playbook_queue.csv  rows {len(q_out):,}")

METRICS = {
    "base_rate": round(BASE_RATE, 4),
    "grouped_auc": round(auc_grp, 4),
    "grouped_p_at_50": round(p_grp_m, 4),
    "grouped_p_at_50_random_ranking": round(p_grp_r, 4),
    "random_auc": round(auc_rand, 4),
    "random_p_at_50": round(p_rand_m, 4),
    "rule_p_at_50": round(p_rule, 4),
    "n_labeled": int(len(labeled)), "n_frame_rows": int(len(frame)),
    "n_clients": int(labeled["client_hash_id"].nunique()),
    "decision_date": DECISION_DATE,
    "feature_window": f"{FEAT_LO}..{FEAT_HI}", "outcome_window": f"{OUT_LO}..{OUT_HI}",
    "model_type": "LogisticRegression(max_iter=500, C=1.0), no scaler, 5 features",
    "split": "GroupKFold(5) by client_hash_id; random row split quoted for contrast only",
    "queue_score": "predicted_probability from a full-frame fit - in-sample; ranking quality is only ever quoted out-of-fold",
    "queue_label": "is_visibility_loss is the March within-month outcome, exported for read-off only - not a future prediction",
    "reason_code_counts": {k: int(v) for k, v in q["reason_code"].value_counts().items()},
    "ctr_split_decline": {"below_median": round(ctr_lo_rate, 4), "above_median": round(ctr_hi_rate, 4)},
    "activity_decline": {"le_10_days": round(act_low_rate, 4), "15_of_15_days": round(act_15_rate, 4)},
    "age_oldest_quartile_decline": round(age_oldest_rate, 4),
    "healthy_looking": {"n": n_healthy, "decline_rate": round(healthy_rate, 4)},
    "source": "re-measured in work/notebooks/capstone.ipynb from the local warehouse cache",
}
(OUT_DIR / "capstone_metrics.json").write_text(json.dumps(METRICS, indent=2) + "\n")
print("wrote work/outputs/capstone_metrics.json")

=== reproducibility of the frame (re-ran the same SQL, same stable ORDER BY) ===
same rows in the same order : True
max abs diff per feature    : f_log_impressions 0.0e+00, f_ctr 0.0e+00, f_avg_position 2.8e-14, f_active_days 0.0e+00, f_log_age_days 0.0e+00
worst difference 2.8e-14 -> within 1e-12? True  (parallel-AVG float noise, not a data change)

wrote work/figures/queue_decline_profile.png  (decile 1 0.1848 -> decile 10 0.5567; the out-of-fold numbers to quote are in §4)


wrote work/figures/reason_code_breakdown.png  (blue = above base rate, grey = at or below;
 the grey bars are the honest half of the figure, not a defect)



wrote work/outputs/action_playbook_queue.csv  rows 77,400
wrote work/outputs/capstone_metrics.json


## Self-check

- [x] Every section filled — markdown thinking **and** the code that backs it
- [x] Runs top to bottom with no errors (the executed outputs are saved in this file)
- [x] No client names, client domains/URLs, or private queries — pseudonymized hashes and aggregates only
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Every headline number re-measured and checked against the deployed paper below

**Data credit:** Built on the [FlyRank ML Internship dataset](https://flyrank.ai) — warehouse `FlyRank/internship-warehouse` (gated, read token; access steps in `skills/flyrank/flyrank-data`).
**Reproducibility:** seeds — `StratifiedKFold(3, shuffle=True, random_state=0)`, P@k random ranking `RandomState(0)`; `GroupKFold(5)` is deterministic given the groups. Full details in paper §7.

In [10]:
# --- Receipts: every headline number in the paper, re-measured in THIS notebook ---
# Stable metrics: asserted. Comparator P@50s: reported with their measured version spread,
# not asserted (seeded permutation / np.argsort tie-breaking on a 3-level rule score).
STABLE = {
    "base_rate": (BASE_RATE, 0.4438),
    "grouped_auc": (auc_grp, 0.5902),
    "grouped_p_at_50": (p_grp_m, 0.4800),
    "random_auc": (auc_rand, 0.6262),
    "random_p_at_50": (p_rand_m, 0.5600),
    "n_labeled": (len(labeled), 77400),
    "n_frame_rows": (len(frame), 151981),
    "n_clients": (labeled["client_hash_id"].nunique(), 38),
    "ctr_below_median_decline": (ctr_lo_rate, 0.5072),
    "ctr_above_median_decline": (ctr_hi_rate, 0.3804),
    "activity_le10_decline": (act_low_rate, 0.1251),
    "activity_15of15_decline": (act_15_rate, 0.4948),
    "age_oldest_quartile_decline": (age_oldest_rate, 0.3707),
    "healthy_looking_n": (n_healthy, 15736),
    "healthy_looking_decline": (healthy_rate, 0.3479),
    "low_ctr_code_n": (int((q["reason_code"] == "low_ctr_more_decline").sum()), 38695),
}
SENSITIVE = {
    "grouped_p_at_50_random_ranking": (p_grp_r, 0.4080),
    "rule_p_at_50": (p_rule, 0.4360),
}
def _fmt(x):
    return f"{x:,}" if isinstance(x, (int, np.integer)) else f"{x:.4f}"
rows = [{"metric": k, "re-measured here": _fmt(m), "in the paper": _fmt(p),
         "match": "yes" if abs(float(m) - float(p)) < 5e-4 else "NO"}
        for k, (m, p) in STABLE.items()]
rows += [{"metric": k + " (version-sensitive)", "re-measured here": _fmt(m), "in the paper": _fmt(p),
          "match": "spread"} for k, (m, p) in SENSITIVE.items()]
receipt = pd.DataFrame(rows)
print(receipt.to_string(index=False))
bad = receipt.loc[receipt["match"] == "NO", "metric"].tolist()
assert not bad, f"stable numbers diverge from the paper: {bad}"
assert all(m < p_grp_m for m, _ in SENSITIVE.values()), "comparators must stay below the model's P@50"
worst_spread = max(abs(m - p) for m, p in SENSITIVE.values())
print(f"\nALL STABLE RECEIPTS MATCH. The two comparator P@50s move by up to {worst_spread:.4f} "
      "with the library's")
print("tie-breaking; both stay below the model's 0.4800, so the paper's conclusion is unchanged.")

                                            metric re-measured here in the paper  match
                                         base_rate           0.4438       0.4438    yes
                                       grouped_auc           0.5902       0.5902    yes
                                   grouped_p_at_50           0.4800       0.4800    yes
                                        random_auc           0.6262       0.6262    yes
                                    random_p_at_50           0.5600       0.5600    yes
                                         n_labeled           77,400       77,400    yes
                                      n_frame_rows          151,981      151,981    yes
                                         n_clients               38           38    yes
                          ctr_below_median_decline           0.5072       0.5072    yes
                          ctr_above_median_decline           0.3804       0.3804    yes
                             act